# Lead-Lag 2026_76 Phase B（バックテスト中心）

- 対象論文: `docs/sources/2026_76.pdf`
- 目的: `PCA SUB` / `MOM` / `PCA PLAIN` / `DOUBLE` の4戦略を日次で再現し、
  `AR` / `RISK` / `R/R` / `MDD` と累積リターンを比較する。
- 注意: 本 notebook では FF3/Carhart 回帰（表3/4相当）は実行しない。


## 1. Setup & Public Parameters

このセルのパラメータを変更すれば、同一ロジックで再実行できる。

In [ ]:
from __future__ import annotations

from pathlib import Path
from dataclasses import dataclass
from typing import Dict, Iterable, List, Tuple

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import yfinance as yf
from IPython.display import display

plt.style.use('ggplot')
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 200)

# ---------------------------------------------------------
# Public parameters (required by plan)
# ---------------------------------------------------------
BASE = Path('/Users/kencharoff/workspace/projects/lead-lag')
NB_NAME = 'lead_lag_2026_76_phase_b'

SAMPLE_START = '2010-01-01'
SAMPLE_END = '2025-12-31'

US_TICKERS = ['XLB','XLC','XLE','XLF','XLI','XLK','XLP','XLRE','XLU','XLV','XLY']
JP_TICKERS = [
    '1617.T','1618.T','1619.T','1620.T','1621.T','1622.T','1623.T','1624.T','1625.T',
    '1626.T','1627.T','1628.T','1629.T','1630.T','1631.T','1632.T','1633.T'
]

LOOKBACK_L = 60
LAMBDA = 0.9
K0 = 3
K = 3
Q = 0.3

AUTO_DOWNLOAD = True
USE_CACHE = True
CACHE_DIR = BASE / 'data' / 'cache' / 'phase_b'

RUN_REGRESSION = False  # 固定OFF（今回範囲外）

assert K0 == 3, 'This implementation assumes K0=3 by design.'
assert K == 3, 'This implementation assumes K=3 by design.'
assert 0 < Q < 0.5

print('BASE:', BASE)
print('NOTEBOOK:', NB_NAME)
print('SAMPLE:', SAMPLE_START, 'to', SAMPLE_END)
print('US tickers:', len(US_TICKERS), US_TICKERS)
print('JP tickers:', len(JP_TICKERS), JP_TICKERS)
print('LOOKBACK_L=', LOOKBACK_L, 'LAMBDA=', LAMBDA, 'K0=', K0, 'K=', K, 'Q=', Q)
print('AUTO_DOWNLOAD=', AUTO_DOWNLOAD, 'USE_CACHE=', USE_CACHE)
print('CACHE_DIR=', CACHE_DIR)
print('RUN_REGRESSION=', RUN_REGRESSION)


## 2. Data I/O Helpers

In [ ]:
REQ_FIELDS = ['Adj Close', 'Close', 'Open']


def _ensure_cache_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def _cache_path(name: str) -> Path:
    return CACHE_DIR / f'{name}_ohlc_{SAMPLE_START}_{SAMPLE_END}.csv'


def _read_multiindex_ohlc_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, header=[0, 1], index_col=0, parse_dates=True)
    df.index.name = 'Date'
    # yfinance保存CSVの列順差異を吸収（Price/Ticker or Ticker/Price）
    lvl0 = set(df.columns.get_level_values(0))
    lvl1 = set(df.columns.get_level_values(1))
    if {'Adj Close', 'Close', 'Open'}.issubset(lvl0):
        out = df.copy()
    elif {'Adj Close', 'Close', 'Open'}.issubset(lvl1):
        out = df.swaplevel(0, 1, axis=1).sort_index(axis=1)
    else:
        raise ValueError(f'Unexpected MultiIndex columns in {path}')
    out = out.sort_index()
    return out


def _normalize_yf_columns(df: pd.DataFrame) -> pd.DataFrame:
    if not isinstance(df.columns, pd.MultiIndex):
        raise ValueError('Expected yfinance DataFrame with MultiIndex columns')

    lvl0 = set(df.columns.get_level_values(0))
    lvl1 = set(df.columns.get_level_values(1))

    if {'Adj Close', 'Close', 'Open'}.issubset(lvl0):
        out = df.copy()
    elif {'Adj Close', 'Close', 'Open'}.issubset(lvl1):
        out = df.swaplevel(0, 1, axis=1)
    else:
        raise ValueError('Could not normalize yfinance MultiIndex columns')

    out = out.sort_index(axis=1)
    out.index = pd.to_datetime(out.index)
    out.index.name = 'Date'
    return out


def load_or_download_ohlc(name: str, tickers: List[str]) -> pd.DataFrame:
    cpath = _cache_path(name)
    if USE_CACHE and cpath.exists():
        print(f'[cache] loading {name}: {cpath}')
        return _read_multiindex_ohlc_csv(cpath)

    if not AUTO_DOWNLOAD:
        raise RuntimeError(
            f'Cache not found for {name} and AUTO_DOWNLOAD=False. Expected: {cpath}'
        )

    print(f'[download] yfinance {name} tickers={len(tickers)}')
    raw = yf.download(
        tickers=tickers,
        start=SAMPLE_START,
        end=SAMPLE_END,
        auto_adjust=False,
        progress=False,
        group_by='column',
        actions=False,
        threads=True,
    )

    if raw is None or len(raw) == 0:
        raise RuntimeError(
            'yfinance download returned empty dataframe. '
            'If running in restricted sandbox, network escalation is required.'
        )

    out = _normalize_yf_columns(raw)

    # 必須フィールドの存在確認
    for f in REQ_FIELDS:
        if f not in out.columns.get_level_values(0):
            raise RuntimeError(f'Missing required field {f} in yfinance output for {name}')

    _ensure_cache_dir(CACHE_DIR)
    out.to_csv(cpath)
    print(f'[cache] saved {name}: {cpath}')
    return out


def extract_panel(ohlc: pd.DataFrame, field: str, tickers: List[str]) -> pd.DataFrame:
    panel = ohlc[field].copy()
    # 未取得ティッカーがある場合は列を補完してNaN埋め
    for t in tickers:
        if t not in panel.columns:
            panel[t] = np.nan
    panel = panel[tickers].sort_index()
    panel.index = pd.to_datetime(panel.index)
    return panel


def ticker_coverage_stats(ohlc: pd.DataFrame, tickers: List[str], market_name: str) -> pd.DataFrame:
    adj = extract_panel(ohlc, 'Adj Close', tickers)
    total_days = len(adj.index)
    rows = []
    for t in tickers:
        nn = int(adj[t].notna().sum())
        miss_rate = float(1.0 - (nn / total_days if total_days > 0 else np.nan))
        rows.append({'market': market_name, 'ticker': t, 'non_null_days': nn, 'total_days': total_days, 'missing_rate': miss_rate})
    df = pd.DataFrame(rows)
    return df


## 3. Download / Cache Load + Data Validation

In [ ]:
us_ohlc = load_or_download_ohlc('us11', US_TICKERS)
jp_ohlc = load_or_download_ohlc('jp17', JP_TICKERS)

cov_us = ticker_coverage_stats(us_ohlc, US_TICKERS, 'US')
cov_jp = ticker_coverage_stats(jp_ohlc, JP_TICKERS, 'JP')
coverage = pd.concat([cov_us, cov_jp], ignore_index=True)

succ_us = (cov_us['non_null_days'] > 0).mean()
succ_jp = (cov_jp['non_null_days'] > 0).mean()

print('US success rate:', f'{succ_us:.2%}', f'({(cov_us.non_null_days > 0).sum()}/{len(US_TICKERS)})')
print('JP success rate:', f'{succ_jp:.2%}', f'({(cov_jp.non_null_days > 0).sum()}/{len(JP_TICKERS)})')

print('\nCoverage head:')
display(coverage.head(10))

print('\nWorst missing-rate tickers:')
display(coverage.sort_values('missing_rate', ascending=False).head(10))


## 4. Adjusted Open/Close and Return Construction (`rcc`, `roc`)

In [ ]:
def build_adjusted_open_close(ohlc: pd.DataFrame, tickers: List[str]) -> Tuple[pd.DataFrame, pd.DataFrame]:
    adj_close = extract_panel(ohlc, 'Adj Close', tickers)
    close = extract_panel(ohlc, 'Close', tickers)
    open_ = extract_panel(ohlc, 'Open', tickers)

    ratio = adj_close / close.replace(0, np.nan)
    adj_open = open_ * ratio

    return adj_open, adj_close


def build_returns(adj_open: pd.DataFrame, adj_close: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    rcc = adj_close / adj_close.shift(1) - 1.0
    roc = adj_close / adj_open - 1.0
    return rcc, roc


us_adj_open, us_adj_close = build_adjusted_open_close(us_ohlc, US_TICKERS)
jp_adj_open, jp_adj_close = build_adjusted_open_close(jp_ohlc, JP_TICKERS)

us_rcc, _ = build_returns(us_adj_open, us_adj_close)
jp_rcc, jp_roc = build_returns(jp_adj_open, jp_adj_close)

# 共通営業日（US情報とJP評価を同日ラベルで接続するため）
common_dates = us_rcc.index.intersection(jp_roc.index)
common_dates = common_dates[(common_dates >= pd.Timestamp(SAMPLE_START)) & (common_dates <= pd.Timestamp(SAMPLE_END))]
common_dates = common_dates.sort_values()

us_rcc_c = us_rcc.reindex(common_dates)
jp_rcc_c = jp_rcc.reindex(common_dates)
jp_roc_c = jp_roc.reindex(common_dates)

all_rcc_c = pd.concat([us_rcc_c, jp_rcc_c], axis=1)
all_cols = US_TICKERS + JP_TICKERS
all_rcc_c = all_rcc_c[all_cols]

print('common trading days:', len(common_dates))
print('all_rcc shape:', all_rcc_c.shape)
print('jp_roc shape:', jp_roc_c.shape)


## 5. Subspace / Signal Helpers (`PCA SUB`, `PCA PLAIN`, `MOM`)

In [ ]:
US_CYCLICAL = {'XLB', 'XLE', 'XLF', 'XLRE'}
US_DEFENSIVE = {'XLK', 'XLP', 'XLU', 'XLV'}
JP_CYCLICAL = {'1618.T', '1625.T', '1629.T', '1631.T'}
JP_DEFENSIVE = {'1617.T', '1621.T', '1627.T', '1630.T'}


def _normalize(v: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    n = float(np.linalg.norm(v))
    if n < eps:
        raise ValueError('vector norm too small')
    return v / n


def _project_out(v: np.ndarray, basis: List[np.ndarray]) -> np.ndarray:
    out = v.copy().astype(float)
    for b in basis:
        out = out - np.dot(out, b) * b
    return out


def build_prior_subspace(us_cols: List[str], jp_cols: List[str]) -> np.ndarray:
    n_u = len(us_cols)
    n_j = len(jp_cols)
    n = n_u + n_j

    # v1: global
    v1 = np.ones(n, dtype=float)
    v1 = _normalize(v1)

    # v2: country spread (+US, -JP) orthogonalized to v1
    v2_raw = np.concatenate([np.ones(n_u), -np.ones(n_j)]).astype(float)
    v2 = _project_out(v2_raw, [v1])
    v2 = _normalize(v2)

    # v3: cyclical(+1), defensive(-1), neutral(0); orthogonalized to v1, v2
    sign = np.zeros(n, dtype=float)
    for i, t in enumerate(us_cols):
        if t in US_CYCLICAL:
            sign[i] = 1.0
        elif t in US_DEFENSIVE:
            sign[i] = -1.0
    offset = len(us_cols)
    for j, t in enumerate(jp_cols):
        if t in JP_CYCLICAL:
            sign[offset + j] = 1.0
        elif t in JP_DEFENSIVE:
            sign[offset + j] = -1.0
    v3 = _project_out(sign, [v1, v2])
    v3 = _normalize(v3)

    V0 = np.column_stack([v1, v2, v3])
    return V0


def safe_corr(z_df: pd.DataFrame, min_periods: int) -> pd.DataFrame:
    c = z_df.corr(min_periods=min_periods)
    c = c.reindex(index=z_df.columns, columns=z_df.columns)
    c = c.fillna(0.0)
    arr = c.to_numpy(copy=True)
    np.fill_diagonal(arr, 1.0)
    return pd.DataFrame(arr, index=c.index, columns=c.columns)


def build_c0(rcc_all: pd.DataFrame, V0: np.ndarray, sample_start: str, sample_end: str) -> np.ndarray:
    sub = rcc_all.loc[sample_start:sample_end]
    # 欠損が多いETFを考慮して pairwise corr を使う
    C_full_df = sub.corr(min_periods=max(20, LOOKBACK_L // 2))
    C_full_df = C_full_df.reindex(index=rcc_all.columns, columns=rcc_all.columns).fillna(0.0)
    C_full = C_full_df.to_numpy(copy=True)
    np.fill_diagonal(C_full, 1.0)

    D0 = np.diag(np.diag(V0.T @ C_full @ V0))
    C0raw = V0 @ D0 @ V0.T

    diag = np.diag(C0raw).copy()
    diag[diag <= 1e-12] = 1.0
    inv_sqrt = np.diag(1.0 / np.sqrt(diag))
    C0 = inv_sqrt @ C0raw @ inv_sqrt
    C0 = (C0 + C0.T) / 2.0
    np.fill_diagonal(C0, 1.0)
    return C0


def topk_bottomk_weights(scores: pd.Series, next_ret: pd.Series, q: float, full_index: List[str]) -> pd.Series:
    eligible = scores.index[scores.notna() & next_ret.notna()]
    n = len(eligible)
    w = pd.Series(0.0, index=full_index)
    if n < 4:
        return w

    k = max(1, int(np.floor(q * n)))
    ranked = scores.loc[eligible].sort_values()

    short_idx = ranked.index[:k]
    long_idx = ranked.index[-k:]

    # 重複回避
    short_set = set(short_idx)
    long_idx = [x for x in long_idx if x not in short_set]
    if len(long_idx) == 0:
        return w

    w.loc[list(short_set)] = -1.0 / len(short_set)
    w.loc[long_idx] = 1.0 / len(long_idx)
    return w


def double_sort_weights(score_mom: pd.Series, score_sub: pd.Series, next_ret: pd.Series, full_index: List[str]) -> pd.Series:
    eligible = score_mom.index[score_mom.notna() & score_sub.notna() & next_ret.notna()]
    w = pd.Series(0.0, index=full_index)
    if len(eligible) < 4:
        return w

    mom_e = score_mom.loc[eligible]
    sub_e = score_sub.loc[eligible]

    med_m = mom_e.median()
    med_s = sub_e.median()

    long_idx = eligible[(mom_e >= med_m) & (sub_e >= med_s)]
    short_idx = eligible[(mom_e < med_m) & (sub_e < med_s)]

    if len(long_idx) == 0 or len(short_idx) == 0:
        return w

    w.loc[long_idx] = 1.0 / len(long_idx)
    w.loc[short_idx] = -1.0 / len(short_idx)
    return w


def metrics_from_returns(ret: pd.Series, annualization: int = 252) -> Dict[str, float]:
    ret = ret.dropna()
    if len(ret) == 0:
        return {'AR': np.nan, 'RISK': np.nan, 'R/R': np.nan, 'MDD': np.nan, 'N': 0}

    ar = ret.mean() * annualization * 100.0
    risk = ret.std(ddof=1) * np.sqrt(annualization) * 100.0
    rr = ar / risk if (risk is not None and np.isfinite(risk) and risk != 0) else np.nan

    wealth = (1.0 + ret).cumprod()
    dd = wealth / wealth.cummax() - 1.0
    mdd = abs(dd.min()) * 100.0

    return {'AR': ar, 'RISK': risk, 'R/R': rr, 'MDD': mdd, 'N': int(len(ret))}


## 6. Main Backtest Engine（時点整合を明示）

In [ ]:
@dataclass
class BacktestResult:
    strategy_returns: Dict[str, pd.Series]
    metrics: pd.DataFrame
    diagnostics: pd.DataFrame
    signal_store: Dict[str, pd.DataFrame]


V0 = build_prior_subspace(US_TICKERS, JP_TICKERS)
C0 = build_c0(all_rcc_c, V0, '2010-01-01', '2014-12-31')


records_diag = []
rets = {'MOM': [], 'PCA_PLAIN': [], 'PCA_SUB': [], 'DOUBLE': []}

signals_sub = {}
signals_plain = {}
signals_mom = {}

# decision_date=t, hold_date=t+1 の対応（共通営業日上で1日先）
for i in range(LOOKBACK_L, len(common_dates) - 1):
    t = common_dates[i]
    t_next = common_dates[i + 1]

    window_dates = common_dates[i - LOOKBACK_L:i]
    assert window_dates.max() < t, 'lookback window leaks future data'
    assert t_next > t, 'hold date must be after decision date'

    window = all_rcc_c.loc[window_dates, all_cols]

    mu = window.mean(skipna=True)
    sigma = window.std(skipna=True, ddof=0).replace(0.0, np.nan)

    z_window = (window - mu) / sigma
    z_window = z_window.replace([np.inf, -np.inf], np.nan)

    C_t_df = safe_corr(z_window, min_periods=max(20, LOOKBACK_L // 2))
    C_t = C_t_df.values

    C_plain = C_t
    C_sub = (1.0 - LAMBDA) * C_t + LAMBDA * C0
    C_sub = (C_sub + C_sub.T) / 2.0
    np.fill_diagonal(C_sub, 1.0)

    # eig (descending)
    eval_p, evec_p = np.linalg.eigh(C_plain)
    idx_p = np.argsort(eval_p)[::-1]
    Vp = evec_p[:, idx_p[:K]]

    eval_s, evec_s = np.linalg.eigh(C_sub)
    idx_s = np.argsort(eval_s)[::-1]
    Vs = evec_s[:, idx_s[:K]]

    # current z_u(t): (rcc_t - mu_t) / sigma_t
    rcc_t = all_rcc_c.loc[t, all_cols]
    z_t = ((rcc_t - mu) / sigma).replace([np.inf, -np.inf], np.nan).fillna(0.0)

    z_u = z_t.loc[US_TICKERS].values.astype(float)

    # plain signal
    Vp_u = Vp[:len(US_TICKERS), :]
    Vp_j = Vp[len(US_TICKERS):, :]
    f_p = Vp_u.T @ z_u
    s_plain = pd.Series(Vp_j @ f_p, index=JP_TICKERS)

    # sub signal
    Vs_u = Vs[:len(US_TICKERS), :]
    Vs_j = Vs[len(US_TICKERS):, :]
    f_s = Vs_u.T @ z_u
    s_sub = pd.Series(Vs_j @ f_s, index=JP_TICKERS)

    # MOM signal: JP rcc rolling mean in window
    s_mom = jp_rcc_c.loc[window_dates, JP_TICKERS].mean(skipna=True)

    signals_plain[t] = s_plain
    signals_sub[t] = s_sub
    signals_mom[t] = s_mom

    next_roc = jp_roc_c.loc[t_next, JP_TICKERS]

    w_mom = topk_bottomk_weights(s_mom, next_roc, Q, JP_TICKERS)
    w_plain = topk_bottomk_weights(s_plain, next_roc, Q, JP_TICKERS)
    w_sub = topk_bottomk_weights(s_sub, next_roc, Q, JP_TICKERS)
    w_double = double_sort_weights(s_mom, s_sub, next_roc, JP_TICKERS)

    for strat, w in [('MOM', w_mom), ('PCA_PLAIN', w_plain), ('PCA_SUB', w_sub), ('DOUBLE', w_double)]:
        gross = float((w * next_roc.fillna(0.0)).sum())
        # 戦略整合チェック: 有効日でのみ strict assert
        if np.abs(w).sum() > 0:
            assert np.isclose(w.sum(), 0.0, atol=1e-10), f'{strat}: sum(w)!=0 on {t}'
            assert np.isclose(np.abs(w).sum(), 2.0, atol=1e-10), f'{strat}: sum(|w|)!=2 on {t}'
        rets[strat].append((t_next, gross))

        records_diag.append({
            'decision_date': t,
            'hold_date': t_next,
            'strategy': strat,
            'n_eligible_roc': int(next_roc.notna().sum()),
            'sum_w': float(w.sum()),
            'sum_abs_w': float(np.abs(w).sum()),
            'active_names': int((np.abs(w) > 0).sum()),
            'return': gross,
        })


ret_series = {}
for k, xs in rets.items():
    s = pd.Series({d: r for d, r in xs}).sort_index()
    s.name = k
    ret_series[k] = s

metrics_rows = []
for k, s in ret_series.items():
    m = metrics_from_returns(s, annualization=252)
    m['Strategy'] = k
    metrics_rows.append(m)

metrics_df = pd.DataFrame(metrics_rows).set_index('Strategy')[['AR', 'RISK', 'R/R', 'MDD', 'N']]
diag_df = pd.DataFrame(records_diag).sort_values(['hold_date', 'strategy'])

signal_store = {
    'MOM': pd.DataFrame(signals_mom).T,
    'PCA_PLAIN': pd.DataFrame(signals_plain).T,
    'PCA_SUB': pd.DataFrame(signals_sub).T,
}

bt = BacktestResult(
    strategy_returns=ret_series,
    metrics=metrics_df,
    diagnostics=diag_df,
    signal_store=signal_store,
)

print('Backtest done.')
print('diagnostics rows:', len(bt.diagnostics))
print('signal rows:', {k: len(v) for k, v in bt.signal_store.items()})


## 7. Validation Checks（計画の Test Plan をコード化）

In [ ]:
# 1) シグナル時点と評価時点の整合（t -> t+1）
assert (bt.diagnostics['hold_date'] > bt.diagnostics['decision_date']).all(), 'Found non-forward mapping.'

# 2) 未来参照チェック（lookback windowの最大日付<t はエンジン内assertで担保済み）
# ここでは各strategyで日時の単調増加を確認
for name, s in bt.strategy_returns.items():
    assert s.index.is_monotonic_increasing, f'{name} index not monotonic increasing'

# 3) 戦略整合（有効日の sum(w)=0, sum(|w|)=2）
active = bt.diagnostics[bt.diagnostics['sum_abs_w'] > 0].copy()
assert np.allclose(active['sum_w'].values, 0.0, atol=1e-10), 'sum(w)!=0 exists'
assert np.allclose(active['sum_abs_w'].values, 2.0, atol=1e-10), 'sum(|w|)!=2 exists'

# 4) 4戦略の主要指標がNaNでない
for col in ['AR', 'RISK', 'R/R', 'MDD']:
    if bt.metrics[col].isna().any():
        raise AssertionError(f'NaN found in metrics column: {col}')

# 5) 共通営業日数・欠損率サマリ
common_days = len(common_dates)
missing_summary = coverage.groupby('market')['missing_rate'].mean().rename('avg_missing_rate')

print('Validation passed.')
print('common trading days:', common_days)
print('avg missing rate by market:')
display(missing_summary.to_frame())


## 8. Performance Table（表2相当）

In [ ]:
paper_table2 = pd.DataFrame(
    {
        'AR': {'MOM': 5.63, 'PCA_PLAIN': 6.24, 'PCA_SUB': 23.79, 'DOUBLE': 18.86},
        'RISK': {'MOM': 10.59, 'PCA_PLAIN': 9.94, 'PCA_SUB': 10.70, 'DOUBLE': 11.16},
        'R/R': {'MOM': 0.53, 'PCA_PLAIN': 0.62, 'PCA_SUB': 2.22, 'DOUBLE': 1.69},
        'MDD': {'MOM': 16.97, 'PCA_PLAIN': 23.65, 'PCA_SUB': 9.58, 'DOUBLE': 12.10},
    }
)
paper_table2 = paper_table2.loc[['MOM', 'PCA_PLAIN', 'PCA_SUB', 'DOUBLE']]

result_tbl = bt.metrics[['AR', 'RISK', 'R/R', 'MDD']].copy()
result_tbl = result_tbl.loc[['MOM', 'PCA_PLAIN', 'PCA_SUB', 'DOUBLE']]

diff_tbl = result_tbl - paper_table2

print('Result table (daily backtest; annualized):')
display(result_tbl.round(4))

print('Reference table (paper Table 2):')
display(paper_table2)

print('Difference (Result - Paper):')
display(diff_tbl.round(4))


## 9. Cumulative Return Plot（図2相当）

In [ ]:
cum = pd.DataFrame({k: (1.0 + v).cumprod() for k, v in bt.strategy_returns.items()}).dropna(how='all')

fig, ax = plt.subplots(figsize=(12, 6))
for c in ['MOM', 'PCA_PLAIN', 'PCA_SUB', 'DOUBLE']:
    if c in cum.columns:
        ax.plot(cum.index, cum[c], label=c)

ax.set_title('Cumulative Returns (Phase B)')
ax.set_xlabel('Date')
ax.set_ylabel('Wealth Index')
ax.legend(loc='best')
plt.tight_layout()
plt.show()


## 10. Diagnostics Snapshot

In [ ]:
print('Diagnostics sample:')
display(bt.diagnostics.head(12))

print('Active-name count summary by strategy:')
display(
    bt.diagnostics.assign(active=bt.diagnostics['active_names'])
    .groupby('strategy')['active']
    .describe()
)


## 11. Notes

- 本 notebook は Phase B「バックテスト中心」の実装。
- `RUN_REGRESSION=False` 固定のため、FF3/Carhart 回帰は今回含めない。
- yfinance がネットワーク制約下で失敗する環境では、実行時に通信権限が必要。